In [7]:
#Create the main Repo list
import os
import pandas as pd
from urllib.parse import urlparse

# === INPUT / OUTPUT ===
SRC_CSV = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\Clone_Status.csv"
OUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026"
OUT_CSV = os.path.join(OUT_DIR, "2_Total_Repo.csv")
os.makedirs(OUT_DIR, exist_ok=True)

# === Load ===
df = pd.read_csv(SRC_CSV, dtype=str).fillna("")
df.columns = [c.strip() for c in df.columns]

# --- Find columns ---
def pick_col(candidates, cols):
    cols_lower = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

clone_col = pick_col(["clone_status"], df.columns)
yml_col   = pick_col(["yml_detected"], df.columns)
url_col   = pick_col(["html_url"], df.columns)

if not clone_col or not yml_col or not url_col:
    missing = [name for name, col in {"clone_status": clone_col, "yml_detected": yml_col, "html_url/htm_url": url_col}.items() if not col]
    raise ValueError(f"Missing required column(s): {', '.join(missing)}")

# --- Normalize "yes" detection ---
def is_yes(x: str) -> bool:
    return str(x).strip().lower() in {"yes", "true", "y", "1"}

filtered = df[ df[clone_col].apply(is_yes) & df[yml_col].apply(is_yes) ].copy()

# --- Build full_name = owner.repo ---
def url_to_full_name(u: str) -> str:
    try:
        path = urlparse(str(u).strip()).path.strip("/")
        if not path:
            return ""
        if path.endswith(".git"):
            path = path[:-4]
        parts = path.split("/")
        if len(parts) >= 2:
            return f"{parts[0]}.{parts[1]}"
        return path
    except Exception:
        return ""

# Ensure full_name is lowercase
filtered["full_name"] = filtered[url_col].apply(url_to_full_name).str.lower()

# --- Keep only URL + full_name ---
out = filtered[[url_col, "full_name"]].rename(columns={url_col: "html_url"})

# Drop duplicates
out = out.drop_duplicates(subset=["html_url"]).reset_index(drop=True)

# Save
out.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(out)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2_Total_Repo.csv (rows=4518)


In [8]:
#Adds the Instru_tests for each repo

In [2]:
import os
import re
import pandas as pd

BASE_DIR   = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026"
MAIN_CSV   = os.path.join(BASE_DIR, "2_Total_Repo.csv")
CI_YML_CSV = os.path.join(BASE_DIR, "1_CI_YML_Instru.csv")
OUT_CSV    = os.path.join(BASE_DIR, "3_Total_Repo.csv")

REQ_MAIN = {"full_name"}
REQ_CI_MIN = {"full_name", "instru_t_ci_signal", "Exec_Env_Style", "Test_Inv_Style", "ci_platform"}

def norm_bool(x) -> bool:
    s = str(x).strip().lower()
    return s in {"1", "true", "t", "yes", "y"}

def merge_labels(series: pd.Series) -> str:
    vals = []
    for v in series.fillna("").astype(str):
        v = v.strip()
        if not v:
            continue
        vals.extend([p.strip() for p in v.split(",") if p.strip()])
    return ",".join(sorted(set(vals), key=str.lower))

def sanitize_col_name(platform: str) -> str:
    p = (platform or "").strip().lower() or "unknown"
    p = re.sub(r"\s+", "_", p)
    p = re.sub(r"[^a-z0-9_]+", "_", p).strip("_") or "unknown"
    return f"{p}"

def drop_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    # keep the first occurrence of any duplicate column name
    return df.loc[:, ~df.columns.duplicated()].copy()

def main():
    # --- Load main ---
    main_df = pd.read_csv(MAIN_CSV, dtype=str).fillna("")
    main_df = drop_duplicate_columns(main_df)
    if "full_name" not in main_df.columns:
        raise ValueError("MAIN_CSV must contain 'full_name'")
    main_df["full_name"] = main_df["full_name"].astype(str).str.strip()

    # --- Load CI YML ---
    ci_df = pd.read_csv(CI_YML_CSV, dtype=str).fillna("")
    ci_df = drop_duplicate_columns(ci_df)

    missing = REQ_CI_MIN - set(ci_df.columns)
    if missing:
        raise ValueError(f"CI_YML_CSV missing columns: {sorted(missing)}")

    ci_df["full_name"] = ci_df["full_name"].astype(str).str.strip()

    # IMPORTANT: ensure ci_platform is a single Series
    ci_platform = ci_df["ci_platform"].astype(str).str.strip()
    ci_platform_norm = ci_platform.str.lower().replace({"": "unknown"})

    # --- Aggregate signals/styles per repo ---
    agg_signal = (
        ci_df.groupby("full_name", dropna=False)
        .agg(
            instru_t_ci_signal=("instru_t_ci_signal", lambda s: any(norm_bool(x) for x in s)),
            Exec_Env_Style=("Exec_Env_Style", merge_labels),
            Test_Inv_Style=("Test_Inv_Style", merge_labels),
        )
        .reset_index()
    )
    agg_signal["instru_t_ci_signal"] = agg_signal["instru_t_ci_signal"].astype(bool)

    # --- CI platform counts (use crosstab, then rename columns safely) ---
    plat_counts = pd.crosstab(ci_df["full_name"], ci_platform_norm).reset_index()

    rename_map = {c: sanitize_col_name(c) for c in plat_counts.columns if c != "full_name"}
    plat_counts = plat_counts.rename(columns=rename_map)

    plat_cols = [c for c in plat_counts.columns if c.startswith("ci_platform__")]
    plat_counts["ci_platform_total"] = plat_counts[plat_cols].sum(axis=1).astype(int)

    # --- Merge into main ---
    out = main_df.merge(agg_signal, on="full_name", how="left")
    out = out.merge(plat_counts, on="full_name", how="left")

    # Defaults for repos with no CI rows
    out["instru_t_ci_signal"] = out["instru_t_ci_signal"].fillna(False).astype(bool)
    out["Exec_Env_Style"] = out["Exec_Env_Style"].fillna("").astype(str)
    out["Test_Inv_Style"] = out["Test_Inv_Style"].fillna("").astype(str)

    # Fill missing platform counts with 0
    for c in plat_cols + ["ci_platform_total"]:
        if c in out.columns:
            out[c] = out[c].fillna(0).astype(int)

    out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Saved: {OUT_CSV} (rows={len(out)})")

if __name__ == "__main__":
    main()


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3_Total_Repo.csv (rows=4518)


In [3]:
#Update the main dataset for Emu_GMD

In [4]:
# Update 3_Total_Repo using repos that already have Exec_Env_Style containing "Emu_GMD"
# in 1_Gradle_GMD_Instru.csv (match by full_name).
#
# For each matched repo:
#   1) Add "Emu_GMD" to main Exec_Env_Style (comma-separated, sorted, de-duped)
#   2) DO NOT modify instru_t_ci_signal
# Output: 4_Total_Repo.csv

import os
import pandas as pd

BASE_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026"
MAIN_CSV = os.path.join(BASE_DIR, "3_Total_Repo.csv")
GMD_CSV  = os.path.join(BASE_DIR, "1_Gradle_GMD_Instru.csv")
OUT_CSV  = os.path.join(BASE_DIR, "4_Total_Repo.csv")

TARGET_LABEL = "Emu_GMD"

def split_labels(s: str) -> list:
    s = "" if s is None else str(s)
    return [p.strip() for p in s.split(",") if p.strip()]

def join_labels(labels: list) -> str:
    uniq = sorted(set(labels), key=str.lower)
    return ",".join(uniq)

def add_label(existing: str, label: str) -> str:
    labels = split_labels(existing)
    labels.append(label)
    return join_labels(labels)

def has_exact_label(s: str, label: str) -> bool:
    toks = [t.strip().lower() for t in split_labels(s)]
    return label.lower() in toks

def main():
    main_df = pd.read_csv(MAIN_CSV, dtype=str).fillna("")
    gmd_df  = pd.read_csv(GMD_CSV, dtype=str).fillna("")

    if "full_name" not in main_df.columns:
        raise ValueError("3_Total_Repo.csv must contain column 'full_name'.")
    if "full_name" not in gmd_df.columns:
        raise ValueError("1_Gradle_GMD_Instru.csv must contain column 'full_name'.")
    if "Exec_Env_Style" not in gmd_df.columns:
        raise ValueError("1_Gradle_GMD_Instru.csv must contain column 'Exec_Env_Style'.")

    main_df["full_name"] = main_df["full_name"].astype(str).str.strip()
    gmd_df["full_name"]  = gmd_df["full_name"].astype(str).str.strip()

    # Ensure Exec_Env_Style exists in main
    if "Exec_Env_Style" not in main_df.columns:
        main_df["Exec_Env_Style"] = ""

    # --- Filter repos that have Emu_GMD detected in Gradle file ---
    gmd_mask = gmd_df["Exec_Env_Style"].astype(str).apply(lambda s: has_exact_label(s, TARGET_LABEL))
    gmd_set = set(gmd_df.loc[gmd_mask, "full_name"].tolist())

    # --- Update Exec_Env_Style only (do NOT touch instru_t_ci_signal) ---
    def update_exec_env_style(row):
        if row["full_name"] in gmd_set:
            return add_label(row.get("Exec_Env_Style", ""), TARGET_LABEL)
        return row.get("Exec_Env_Style", "")

    main_df["Exec_Env_Style"] = main_df.apply(update_exec_env_style, axis=1)

    main_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Saved: {OUT_CSV} (rows={len(main_df)}; repos_with_{TARGET_LABEL}={len(gmd_set)})")

if __name__ == "__main__":
    main()


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\4_Total_Repo.csv (rows=4518; repos_with_Emu_GMD=24)


In [5]:
#Append Metadata from 2_Meta_Instru.csv to 4_Total_Repo.csv

In [6]:
# -*- coding: utf-8 -*-
# Left-join Project_Metadata onto 4_Total_Repo (ICST2026) without duplicating column names
# Output: 5_Total_Repo.csv in the SAME ICST2026 folder

import os
import pandas as pd
from datetime import datetime
from urllib.parse import urlparse

BASE_DIR   = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026"
MAIN_PATH  = os.path.join(BASE_DIR, "4_Total_Repo.csv")
META_PATH  = os.path.join(BASE_DIR, "Project_Metadata.csv")
OUT_PATH   = os.path.join(BASE_DIR, "5_Total_Repo.csv")

# Use the same fetch date you used in your study (keep/change as needed)
FETCH_DATE = datetime(2025, 8, 10)

# Columns from metadata to explicitly exclude
EXCLUDE_META_COLS = {
    "html_url", "repo_index", "repo_name", "id", "name", "full_name", "owner"
}

def norm_key_main(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
              .str.replace("/", ".", regex=False)
              .str.strip()
              .str.lower()
    )

def owner_repo_from_html_url(url: str) -> tuple[str, str]:
    try:
        path = urlparse(str(url)).path.strip("/")
        parts = path.split("/")
        if len(parts) >= 2:
            return parts[0].strip(), parts[1].strip()
    except Exception:
        pass
    return "", ""

def meta_key_from_html(series: pd.Series) -> pd.Series:
    return series.astype(str).map(lambda u: ".".join(owner_repo_from_html_url(u))).str.strip().str.lower()

def drop_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    # Keep first occurrence if there are duplicate header names
    return df.loc[:, ~df.columns.duplicated()].copy()

# --- Load ---
main = pd.read_csv(MAIN_PATH, dtype=str).fillna("")
meta = pd.read_csv(META_PATH, dtype=str).fillna("")
main = drop_duplicate_columns(main)
meta = drop_duplicate_columns(meta)

# --- Validate ---
if "full_name" not in main.columns:
    raise ValueError("Main file must contain a 'full_name' column.")
if "html_url" not in meta.columns:
    raise ValueError("Metadata file must contain an 'html_url' column.")

# --- Keys ---
main["__key__"] = norm_key_main(main["full_name"])
meta["__key__"] = meta_key_from_html(meta["html_url"])

# --- repo_age ---
if "created_at" in meta.columns:
    created_dt = pd.to_datetime(meta["created_at"], errors="coerce", utc=True)
    meta["repo_age"] = (
        (pd.to_datetime(FETCH_DATE) - created_dt.dt.tz_localize(None)).dt.days / 365.25
    ).round(2)
else:
    meta["repo_age"] = ""

# --- Filter metadata columns (avoid duplicates + exclusions) ---
main_cols_set = set(main.columns)
meta_cols_to_add = [
    c for c in meta.columns
    if c not in main_cols_set and c != "__key__" and c not in EXCLUDE_META_COLS
]

meta_to_merge = meta[["__key__"] + meta_cols_to_add]

# --- Merge ---
out = (
    main.merge(meta_to_merge, on="__key__", how="left")
        .drop(columns=["__key__"], errors="ignore")
)

# --- Save ---
out.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"Saved: {OUT_PATH} (rows={len(out)}, metadata_cols_added={len(meta_cols_to_add)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_Total_Repo.csv (rows=4518, metadata_cols_added=30)


In [7]:
# adds the Fluter Topic Presence

In [8]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026")
IN_CSV  = BASE_DIR / "5_Total_Repo.csv"
OUT_CSV = BASE_DIR / "5_Total_Repo.csv"

if not IN_CSV.exists():
    raise FileNotFoundError(f"Input CSV not found: {IN_CSV}")

df = pd.read_csv(IN_CSV, dtype=str).fillna("")

if "topics" not in df.columns:
    raise KeyError("Column 'topics' not found in the dataset. Please check the column name.")

def has_flutter_topic(topics: str) -> bool:
    toks = [t.strip().lower() for t in str(topics).split(",") if t.strip()]
    return any(t == "flutter" or t.startswith("flutter-") for t in toks)

# Add as the LAST column
df["flutter_topic_present"] = df["topics"].apply(has_flutter_topic)

df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved: {OUT_CSV}  (rows={len(df)})")
print("flutter_topic_present=True:", int(df["flutter_topic_present"].sum()))


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_Total_Repo.csv  (rows=4518)
flutter_topic_present=True: 550


In [9]:
#add the index column as first column to lockdown the rows for reproducibility of the random sampling.
from pathlib import Path
import pandas as pd

BASE_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026")
IN_CSV   = BASE_DIR / "5_Total_Repo.csv"
OUT_CSV  = BASE_DIR / "5_Total_Repo.csv"

INDEX_COL = "Random_Seed42"  # fixed for reproducibility

if not IN_CSV.exists():
    raise FileNotFoundError(f"Input CSV not found: {IN_CSV}")

df = pd.read_csv(IN_CSV, dtype=str).fillna("")

# Add index column as FIRST column (1-based)
if INDEX_COL in df.columns:
    df = df.drop(columns=[INDEX_COL])

df.insert(0, INDEX_COL, range(1, len(df) + 1))

df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved: {OUT_CSV} (rows={len(df)})")
print(f"Added first column: {INDEX_COL}")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_Total_Repo.csv (rows=4518)
Added first column: Random_Seed42
